In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- bench_precip_pct ---
DF_PRECIP_PD = pd.DataFrame({"value": [1.5, 0.0, np.nan, 4.1, 0.0]})
DF_PRECIP_PL = pl.from_pandas(DF_PRECIP_PD)
df = DF_PRECIP_PD

# --- bench_precip_regular ---
class _ValuesAll:
    def __init__(self, frame): self.df = frame
    def all(self): return self
class _Request:
    def __init__(self, frame): self.values = _ValuesAll(frame)
class _Stations:
    def __init__(self, frame): self._frame = frame
    def filter_by_distance(self, latlon, distance): return _Request(self._frame)

REGULAR_PD = pd.DataFrame({"station_id": ["01048", "02000", "01048"], "value": [1.0, 2.0, np.nan], "date": pd.to_datetime(["2020-01-01", "2020-01-01", "2020-01-02"])})
REGULAR_PL = pl.from_pandas(REGULAR_PD)
FIX_BENCH_PRECIP_REGULAR_STATIONS_PD = _Stations(REGULAR_PD)
FIX_BENCH_PRECIP_REGULAR_STATIONS_PL = _Stations(REGULAR_PL)
FIX_BENCH_PRECIP_REGULAR_STATIONS = FIX_BENCH_PRECIP_REGULAR_STATIONS_PL

print("✅ Fixtures loaded")


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_bench_precip_pct():
    total_amount = len(df["value"])
    zero_amount = len(df[df["value"] == 0.0])
    return zero_amount

def before_bench_precip_regular(stations):
    def get_regular_df(start_date, end_date, exclude_stations):
        request = stations.filter_by_distance(latlon=(50.0, 8.9), distance=30)
        df = request.values.all().df.dropna()
        station_ids = df.station_id.tolist()
        first_station_id = set(station_ids).difference(set(exclude_stations)).pop()
        return df[df["station_id"] == first_station_id]
    return get_regular_df

In [ ]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_bench_precip_pct():
    total_amount = len(df["value"])
    zero_amount = len(df.filter(pl.col("value") == 0.0))
    return zero_amount

def gen_bench_precip_regular(stations):
    def get_regular_df(start_date, end_date, exclude_stations):
        request = stations.filter_by_distance(latlon=(50.0, 8.9), distance=30)
        df = request.values.all().df.drop_nulls()
        station_ids = df["station_id"].to_list()
        first_station_id = set(station_ids).difference(set(exclude_stations)).pop()
        return df.filter(pl.col("station_id") == first_station_id)
    return get_regular_df

In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: bench_precip_regular ===

try:
    _fn = gen_bench_precip_regular(FIX_BENCH_PRECIP_REGULAR_STATIONS_PL)
    _r = _fn("2020-01-01", "2020-01-02", exclude_stations=["02000"])
    print("✅ L1 smoke gen_bench_precip_regular: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_bench_precip_regular: {type(_e).__name__}: {_e}")

try:
    _fn = before_bench_precip_regular(FIX_BENCH_PRECIP_REGULAR_STATIONS_PD)
    _rb = _fn("2020-01-01", "2020-01-02", exclude_stations=["02000"])
    print("✅ L1 smoke before_bench_precip_regular: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_bench_precip_regular: {type(_e).__name__}: {_e}")

try:
    _rb = before_bench_precip_regular(FIX_BENCH_PRECIP_REGULAR_STATIONS_PD)("2020-01-01", "2020-01-02", exclude_stations=["02000"])
    _rg = gen_bench_precip_regular(FIX_BENCH_PRECIP_REGULAR_STATIONS_PL)("2020-01-01", "2020-01-02", exclude_stations=["02000"])
    compare(_rb, _rg, "bench_precip_regular")
except Exception as _e:
    print(f"❌ L2 equivalence bench_precip_regular: setup error — {type(_e).__name__}: {_e}")

# AUDIT-54: compare the exclude-first branch with the oracle.
try:
    _rb = before_bench_precip_regular(FIX_BENCH_PRECIP_REGULAR_STATIONS_PD)("2020-01-01", "2020-01-02", exclude_stations=["01048"])
    _rg = gen_bench_precip_regular(FIX_BENCH_PRECIP_REGULAR_STATIONS_PL)("2020-01-01", "2020-01-02", exclude_stations=["01048"])
    compare(_rb, _rg, "L3 edge bench_precip_regular exclude first oracle", check_row_order=True)
except Exception as _e:
    print(f"❌ L3 edge bench_precip_regular exclude first oracle: {type(_e).__name__}: {_e}")
